<a href="https://colab.research.google.com/github/AfraazUlHaque/ASSIGNMENTS/blob/main/t3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("data.csv")

In [3]:
df.head()

,Country,Density\n(P/Km2),Abbreviation,Agricultural Land( %),Land Area(Km2),Armed Forces size,Birth Rate,Calling Code,Capital/Major City,Co2-Emissions,...,Out of pocket health expenditure,Physicians per thousand,Population,Population: Labor force participation (%),Tax revenue (%),Total tax rate,Unemployment rate,Urban_population,Latitude,Longitude
0,Afghanistan,60,AF,58.10%,"652,230","323,000",32.49,93.0,Kabul,"8,672",...,78.40%,0.28,"38,041,754",48.90%,9.30%,71.40%,11.12%,"9,797,273",33.939110,67.709953
1,Albania,105,AL,43.10%,"28,748","9,000",11.78,355.0,Tirana,"4,536",...,56.90%,1.20,"2,854,191",55.70%,18.60%,36.60%,12.33%,"1,747,593",41.153332,20.168331
2,Algeria,18,DZ,17.40%,"2,381,741","317,000",24.28,213.0,Algiers,"150,006",...,28.10%,1.72,"43,053,054",41.20%,37.20%,66.10%,11.70%,"31,510,100",28.033886,1.659626
3,Andorra,164,AD,40.00%,468,NaN,7.20,376.0,Andorra la Vella,469,...,36.40%,3.33,"77,142",NaN,NaN,NaN,NaN,"67,873",42.506285,1.521801
4,Angola,26,AO,47.50%,"1,246,700","117,000",40.73,244.0,Luanda,"34,693",...,33.40%,0.21,"31,825,295",77.50%,9.20%,49.10%,6.89%,"21,061,025",-11.202692,17.873887


In [5]:
import numpy as np
from sklearn.preprocessing import StandardScaler

df['Co2-Emissions'] = df['Co2-Emissions'].astype(str).str.replace(',', '', regex=False)
df['Co2-Emissions'] = pd.to_numeric(df['Co2-Emissions'], errors='coerce')


df_cleaned = df.dropna(subset=['Co2-Emissions']).copy()


scaler = StandardScaler()
df_cleaned['Co2-Emissions_zscore'] = scaler.fit_transform(df_cleaned[['Co2-Emissions']])

highest_co2_zscore_country = df_cleaned.loc[df_cleaned['Co2-Emissions_zscore'].idxmax()]

print("Country with the highest z-score for 'Co2-Emissions' after standardisation:")
display(highest_co2_zscore_country[['Country', 'Co2-Emissions', 'Co2-Emissions_zscore']])

Country with the highest z-score for 'Co2-Emissions' after standardisation:


,36
Country,China
Co2-Emissions,9893038.0
Co2-Emissions_zscore,11.613369


In [9]:

df['Life expectancy'] = df['Life expectancy'].astype(str).str.replace(',', '', regex=False)
df['Life expectancy'] = pd.to_numeric(df['Life expectancy'], errors='coerce')


df_cleaned_le = df.dropna(subset=['Life expectancy']).copy()


scaler_le = StandardScaler()
df_cleaned_le['Life expectancy_zscore'] = scaler_le.fit_transform(df_cleaned_le[['Life expectancy']])


lowest_le_zscore_country = df_cleaned_le.loc[df_cleaned_le['Life expectancy_zscore'].idxmin()]

print("Country with the lowest z-score for 'Life expectancy' after standardisation:")
display(lowest_le_zscore_country[['Country', 'Life expectancy', 'Life expectancy_zscore']])


if not lowest_le_zscore_country.empty:
    country_name = lowest_le_zscore_country['Country']
    z_score = lowest_le_zscore_country['Life expectancy_zscore']
    life_expectancy = lowest_le_zscore_country['Life expectancy']

    if z_score < 0:
        interpretation = f"This indicates that {country_name} has a life expectancy ({life_expectancy}) significantly below the global average, as its z-score is strongly negative."
    elif z_score == 0:
        interpretation = f"This indicates that {country_name} has a life expectancy ({life_expectancy}) exactly at the global average."
    else:
        interpretation = f"This indicates that {country_name} has a life expectancy ({life_expectancy}) above the global average."
    print(f"\nWhat this output tells you about {country_name}: {interpretation}")
else:
    print("Could not find a country with the lowest life expectancy z-score.")

Country with the lowest z-score for 'Life expectancy' after standardisation:


,33
Country,Central African Republic
Life expectancy,52.8
Life expectancy_zscore,-2.609949



What this output tells you about Central African Republic: This indicates that Central African Republic has a life expectancy (52.8) significantly below the global average, as its z-score is strongly negative.


In [17]:
from sklearn.preprocessing import StandardScaler
from scipy.spatial.distance import euclidean

numeric_cols = [
    'Density\n(P/Km2)', 'Land Area(Km2)', 'Armed Forces size', 'Birth Rate',
    'Calling Code', 'Co2-Emissions', 'Forested Area (%)', 'Gasoline Price', 'GDP',
    'Gross primary education enrollment (%)', 'Gross tertiary education enrollment (%)',
    'Infant mortality', 'Life expectancy', 'Maternal mortality ratio',
    'Physicians per thousand', 'Population', 'Population: Labor force participation (%)',
    'Tax revenue (%)', 'Total tax rate', 'Unemployment rate', 'Urban_population',
    'Latitude', 'Longitude'
]

df_processed = df.copy()

for col in numeric_cols:
    if col in df_processed.columns:
        # Remove commas and '%' and convert to numeric,s
        df_processed[col] = df_processed[col].astype(str).str.replace(',', '', regex=False).str.replace('%', '', regex=False)
        df_processed[col] = pd.to_numeric(df_processed[col], errors='coerce')


for col in numeric_cols:
    if col in df_processed.columns and df_processed[col].isnull().any():
        df_processed[col] = df_processed[col].fillna(df_processed[col].mean())

df_processed_cleaned_for_standardization = df_processed.copy()


final_columns_to_standardize = [col for col in numeric_cols if col in df_processed_cleaned_for_standardization.columns and not df_processed_cleaned_for_standardization[col].isnull().all()]


scaler_all = StandardScaler()
df_processed_cleaned_for_standardization[final_columns_to_standardize] = scaler_all.fit_transform(df_processed_cleaned_for_standardization[final_columns_to_standardize])

columns_to_standardize = final_columns_to_standardize


if 'India' not in df_processed_cleaned_for_standardization['Country'].values:
    print("Error: India not found in the processed dataset.")

if 'China' not in df_processed_cleaned_for_standardization['Country'].values:
    print("Error: China not found in the processed dataset.")

india_features = df_processed_cleaned_for_standardization[df_processed_cleaned_for_standardization['Country'] == 'India'][columns_to_standardize].iloc[0]
china_features = df_processed_cleaned_for_standardization[df_processed_cleaned_for_standardization['Country'] == 'China'][columns_to_standardize].iloc[0]

euclidean_distance_india_china = euclidean(india_features, china_features)
print(f"Euclidean distance between India and China: {euclidean_distance_india_china:.2f}")


other_countries = df_processed_cleaned_for_standardization[df_processed_cleaned_for_standardization['Country'] != 'India']
distances = {}
for index, row in other_countries.iterrows():
    country_name = row['Country']
    features = row[columns_to_standardize]
    distances[country_name] = euclidean(india_features, features)

sorted_distances = sorted(distances.items(), key=lambda item: item[1])
nearest_neighbours = sorted_distances[:3]

print("\nIndia's 3 nearest neighbours (Euclidean distance):")
for country, dist in nearest_neighbours:
    print(f"- {country}: {dist:.2f}")

print("\nWhich group of countries is most likely to appear as India's top 3 nearest neighbours?")
print("Based on the output, the most likely group would be countries with similar developmental, demographic, and economic profiles to India, often found within Asia or other developing regions with large populations.")

Euclidean distance between India and China: 11.39

India's 3 nearest neighbours (Euclidean distance):
- United States: 11.26
- China: 11.39
- Indonesia: 11.54

Which group of countries is most likely to appear as India's top 3 nearest neighbours?
Based on the output, the most likely group would be countries with similar developmental, demographic, and economic profiles to India, often found within Asia or other developing regions with large populations.


In [19]:

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA



print(df.head())


df_numeric = df.select_dtypes(include=[np.number])


df_numeric = df_numeric.dropna()


scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_numeric)


pca = PCA()
X_pca = pca.fit_transform(X_scaled)


explained_var = pca.explained_variance_ratio_

print("\nExplained Variance Ratio:")
print(explained_var)


cumulative_var = np.cumsum(explained_var)

print("\nCumulative Explained Variance:")
print(cumulative_var)

for i, var in enumerate(explained_var[:5]):
    print(f"PC{i+1}: {var:.4f}")

       Country Density\n(P/Km2) Abbreviation Agricultural Land( %)  \
0  Afghanistan               60           AF                58.10%   
1      Albania              105           AL                43.10%   
2      Algeria               18           DZ                17.40%   
3      Andorra              164           AD                40.00%   
4       Angola               26           AO                47.50%   

  Land Area(Km2) Armed Forces size  Birth Rate  Calling Code  \
0        652,230           323,000       32.49          93.0   
1         28,748             9,000       11.78         355.0   
2      2,381,741           317,000       24.28         213.0   
3            468               NaN        7.20         376.0   
4      1,246,700           117,000       40.73         244.0   

  Capital/Major City Co2-Emissions  ... Out of pocket health expenditure  \
0              Kabul         8,672  ...                           78.40%   
1             Tirana         4,536  ...   

In [24]:
import numpy as np


standardized_features = df_processed_cleaned_for_standardization[columns_to_standardize]


means = standardized_features.mean()
stds = standardized_features.std()


means_approx_zero = np.all(np.isclose(means, 0, atol=1e-9))
stds_approx_one = np.all(np.isclose(stds, 1, atol=1e-9))

print(f"All means ≈ 0: {means_approx_zero}")
print(f"All stds ≈ 1: {stds_approx_one}")


All means ≈ 0: False
All stds ≈ 1: False
